In [ ]:
from google.colab import files
uploaded = files.upload()  # Opens a dialog box to upload files

Saving Combined .csv to Combined .csv


In [ ]:
import pandas as pd
df=pd.read_csv('Combined .csv')

In [ ]:
# Ab specific labels ko 'TRUE' ya 'FAKE' main convert karain
df['Label'] = df['Label'].replace({
    'TRUE': 'TRUE',
    'TRUE ': 'TRUE',
    'FAKE': 'FAKE',
    'FAKE ': 'FAKE',
})

In [ ]:
df.drop(columns=['Sr. No.'],inplace=True)

In [ ]:
 #Filter out rows where the column has value 'label'
df = df[df['Label'] != 'Label']
print(df)

                                              News Items Label
0      ٹی ٹی پی نے پنجاب حکومت کے ہیلی کاپٹر کے عملے ...  FAKE
1              مارک زکربرگ سیاست میں آنے کا سوچ رہے ہیں۔  FAKE
2         فریدہ جلال نے اپنی موت کی افواہوں پر تنقید کی۔  FAKE
3      جعلی خبریں: پاپ اسٹار حدیقہ کیانی نے جعلی منشی...  FAKE
4      صنم ماروی نے میڈیا پر گردش کرنے والی زیادتی او...  FAKE
...                                                  ...   ...
10079  بے روزگاری الاؤنس، کیش بیک اسکیمیں - کانگریس ن...  TRUE
10080  چدمبرم نے بجٹ پر تنقید کی، حکومت کا کہنا ہے کہ...  TRUE
10081  نتیش نے بہار میں 16,443 کلومیٹر طویل انسانی زن...  TRUE
10082  بھارت کے نوجوان معروف سوشل میڈیا اسٹار اور کام...  TRUE
10083  کراچی: ’’صحیح وقت پر شادی کر لی ہوتی تو دونوں ...  TRUE

[10083 rows x 2 columns]


In [ ]:
 #Filter out rows where the column has value 'label'
df = df[df['Label'] != 'Label']
print(df)

                                              News Items Label
0      ٹی ٹی پی نے پنجاب حکومت کے ہیلی کاپٹر کے عملے ...  FAKE
1              مارک زکربرگ سیاست میں آنے کا سوچ رہے ہیں۔  FAKE
2         فریدہ جلال نے اپنی موت کی افواہوں پر تنقید کی۔  FAKE
3      جعلی خبریں: پاپ اسٹار حدیقہ کیانی نے جعلی منشی...  FAKE
4      صنم ماروی نے میڈیا پر گردش کرنے والی زیادتی او...  FAKE
...                                                  ...   ...
10079  بے روزگاری الاؤنس، کیش بیک اسکیمیں - کانگریس ن...  TRUE
10080  چدمبرم نے بجٹ پر تنقید کی، حکومت کا کہنا ہے کہ...  TRUE
10081  نتیش نے بہار میں 16,443 کلومیٹر طویل انسانی زن...  TRUE
10082  بھارت کے نوجوان معروف سوشل میڈیا اسٹار اور کام...  TRUE
10083  کراچی: ’’صحیح وقت پر شادی کر لی ہوتی تو دونوں ...  TRUE

[10083 rows x 2 columns]


In [ ]:
#text normalization
import re

def manual_normalize_text(text):
    # Remove diacritics (zabar, zer, pesh)
    text = re.sub(r'[\u064B-\u065F]', '', text)
    # Remove non-Urdu characters (punctuation, digits, special symbols)
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply normalization to the 'News Items' column
df['News Items'] = df['News Items'].apply(manual_normalize_text)
print(df.head())

                                          News Items Label
0  ٹی ٹی پی نے پنجاب حکومت کے ہیلی کاپٹر کے عملے ...  FAKE
1          مارک زکربرگ سیاست میں آنے کا سوچ رہے ہیں۔  FAKE
2     فریدہ جلال نے اپنی موت کی افواہوں پر تنقید کی۔  FAKE
3  جعلی خبریں پاپ اسٹار حدیقہ کیانی نے جعلی منشیا...  FAKE
4  صنم ماروی نے میڈیا پر گردش کرنے والی زیادتی او...  FAKE


In [ ]:
#Tokenization step:convert text into tokens or words
from transformers import AutoTokenizer
#loading the tokenizer
Tokenizer=AutoTokenizer.from_pretrained('distilbert-base-multilingual-cased')

#convert text into tokens
tokens=Tokenizer(df['News Items'].tolist(),
                 max_length=512,
                 add_special_tokens=True,
                 padding=True,
                 truncation=True,
                 return_tensors='pt')
#cheching the tokenized data
print(tokens.keys())


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

dict_keys(['input_ids', 'attention_mask'])


In [ ]:
#step3:loading the bert model
from transformers import AutoModelForSequenceClassification
#loading the bert model for ake news detection
model=AutoModelForSequenceClassification.from_pretrained('distilbert-base-multilingual-cased',
                                                         num_labels=2 #for fake and real
                                                         )
 #Dropout ko set karna
model.config.hidden_dropout_prob = 0.3  # Default = 0.1, increase to reduce overfitting
model.config.attention_probs_dropout_prob = 0.3 # Attention layer dropout

print("Hidden Dropout:", model.config.hidden_dropout_prob)
print("Attention Dropout:", model.config.attention_probs_dropout_prob)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Hidden Dropout: 0.3
Attention Dropout: 0.3


In [ ]:
# step4"Dataloader:making ready the data for training
#now converting the dataset into pytorch dataset
import torch
from torch.utils.data import TensorDataset
#converting the maked tokens into tensores becase te model only take inputs in tensore
# Convert 'Label' column to numerical labels (0 and 1)
label_mapping = {'FAKE': 0, 'TRUE': 1}  # Map 'FAKE' to 0 and 'TRUE' to 1
numerical_labels = df['Label'].map(label_mapping).values

dataset = TensorDataset(tokens['input_ids'],
                        tokens['attention_mask'],
                        torch.tensor(numerical_labels, dtype=torch.int64)) # Specify dtype for numerical labels

In [ ]:
#step5:traintest split
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
#train test split\
train_size=0.8
train_dataset,val_dataset=train_test_split(dataset,train_size=train_size)

#now use the dataloader to nak sure the model training in batches to prevent any cpu crashes or slow processing
train_dataloader=DataLoader(train_dataset,batch_size=16,shuffle=True)
val_dataloader=DataLoader(val_dataset,batch_size=16,shuffle=True)


In [ ]:
# step 6:in here we set the optimizer for weight updated and lr shdular to update the learning rate
# get_shedulaer adjust the learning rate during training
import torch
from torch.optim import AdamW # Import AdamW from torch.optim
from transformers import get_scheduler
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
#optemizer
optimizer=AdamW(model.parameters(),lr=2.7523725716977022e-05,weight_decay=0.00011426159958462337)

#learning rate shedular
lr_shedular=get_scheduler(
    "linear",optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_dataloader)*3
)

In [ ]:
!pip install optuna
!pip install transformers
import optuna
import torch
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification

# Define Objective Function for Optuna
def objective(trial):
    # Load Pre-trained Model (Fine-tuning Required)
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-multilingual-cased",
        num_labels=2
    )

    # Hyperparameter Search Space
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 6e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])

    # Training Arguments
    training_args = TrainingArguments(
        output_dir="./results",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=3,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir="./logs",
        logging_steps=100,
        run_name="bert-finetuning-run"
    )

    # Trainer Initialization
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=lambda features: dict(
            input_ids=torch.stack([f[0] for f in features]),
            attention_mask=torch.stack([f[1] for f in features]),
            labels=torch.tensor([f[2] for f in features], dtype=torch.int64),
        ),
    )

    # Train the Model
    trainer.train()

    # Evaluate Performance
    eval_result = trainer.evaluate()

    return eval_result["eval_loss"]  # Minimizing Loss

# Optuna Study Initialization
study = optuna.create_study(direction="minimize",sampler=optuna.samplers.TPESampler())

# Optimize Objective Function
study.optimize(objective, n_trials=10)  # Running 10 Trials

# Print Best Hyperparameters
print("Best Hyperparameters:", study.best_params)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 24.0 MB/s eta 0:00:00


[I 2025-04-20 07:37:40,069] A new study created in memory with name: no-name-26186bb1-b8ca-4a0f-a9a6-59a5d4565945
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


...

Epoch,Training Loss,Validation Loss
1,0.305100,0.261909
2,0.234700,0.247764
3,0.164400,0.276774


[I 2025-04-20 11:00:16,484] Trial 8 finished with value: 0.2767740488052368 and parameters: {'learning_rate': 1.5016127513487214e-05, 'weight_decay': 0.006132738713870308, 'batch_size': 16}. Best is trial 4 with value: 0.241950124502182.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.324000,0.243241
2,0.192200,0.215226
3,0.082900,0.279582


[I 2025-04-20 11:21:44,487] Trial 9 finished with value: 0.27958187460899353 and parameters: {'learning_rate': 5.3222358054964767e-05, 'weight_decay': 0.0062222584224979225, 'batch_size': 32}. Best is trial 4 with value: 0.241950124502182.


Best Hyperparameters: {'learning_rate': 1.384343268530975e-05, 'weight_decay': 0.00012274018440925597, 'batch_size': 32}
